# hidroxmx-forecasting - Colab entrypoint

Runs one stage of the Paper 2 pipeline on Google Colab with a time-limited GPU.

The notebook (i) clones this repository, (ii) installs the package with GPU-friendly extras, (iii) mounts Google Drive to expose the `hidroxai-mx` snapshot (raw hydrometric + climatological series live there — the R2 mirror stores them in DVC's content-addressable layout that is not addressable by logical path yet), (iv) reads R2 credentials from Colab secrets or the environment, (v) restores `last.ckpt` for the requested `run-id` from R2 when present, (vi) runs exactly one script from `scripts/`, (vii) flushes new checkpoints and metrics to R2 and (viii) prints an exact resume command.

If the GPU session times out, re-open the notebook and re-run: the entrypoint reads the last checkpoint from R2 and continues within `checkpoint_every_n_steps` of where it stopped.

## 1. Verify GPU

Runtime -> Change runtime type -> GPU (T4 is fine). If `nvidia-smi` errors here, switch the runtime before continuing — CPU-only runs finish but defeat the point of this notebook.

In [ ]:
!nvidia-smi | head -20

## 2. Clone the repository

Re-clones only when running fresh — the notebook is idempotent.

In [ ]:
%%bash --no-raise-error
if [ ! -d /content/hidroxmx-forecasting ]; then
    git clone --depth 1 https://github.com/pantrok/hidroxmx-forecasting.git /content/hidroxmx-forecasting
fi
cd /content/hidroxmx-forecasting && git pull --ff-only && git rev-parse --short HEAD

## 3. Install the package

`torch` already ships with the Colab GPU image; we only install this package and the light extras. If a fresh `torch` is needed, uncomment the second line and pin the CUDA build that matches the runtime.

In [ ]:
!pip install -q -e /content/hidroxmx-forecasting[geo,uq]
# !pip install -q --index-url https://download.pytorch.org/whl/cu121 torch==2.3.1

## 4. Mount Google Drive to access the `hidroxai-mx` snapshot

Set `HIDROXAI_MX_DRIVE_PATH` to the folder that holds `hidroxai-mx/data/` on **your** Google Drive. The default matches the layout used by the project owner; adjust if your Drive is organised differently.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

HIDROXAI_MX_DRIVE_PATH = '/content/drive/MyDrive/HidroMx/Data/Código/hidroxai-mx'
os.environ['HIDROXAI_MX_ROOT'] = HIDROXAI_MX_DRIVE_PATH

assert os.path.isdir(os.path.join(HIDROXAI_MX_DRIVE_PATH, 'data')), (
    f'expected hidroxai-mx snapshot under {HIDROXAI_MX_DRIVE_PATH}/data — '
    'edit HIDROXAI_MX_DRIVE_PATH above'
)
print('snapshot root ok:', HIDROXAI_MX_DRIVE_PATH)

## 5. R2 credentials

Store `R2_ENDPOINT_URL`, `R2_BUCKET`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `R2_DATASET_PREFIX`, `R2_PAPER2_PREFIX` as Colab secrets (`Tools -> Secrets`) with notebook access. R2 uploads are optional — pass `--upload-to-r2` to mirror checkpoints and metrics; leave it off for smoke runs.

In [ ]:
from google.colab import userdata
for k in (
    'R2_ENDPOINT_URL', 'R2_BUCKET', 'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY',
    'R2_DATASET_PREFIX', 'R2_PAPER2_PREFIX',
):
    try:
        val = userdata.get(k)
        if val:
            os.environ[k] = val
    except Exception:
        pass

have_r2 = bool(os.environ.get('R2_ENDPOINT_URL'))
print('R2 credentials present:', have_r2)

## 6. Configure the run

One run per invocation — a GPU timeout is bounded to a single stage. Adjust the parameters below and re-run the last cell to iterate.

In [ ]:
STAGE   = '11_train_forecaster'   # any script under scripts/, without .py
RUN_ID  = 'F0-alto-lerma-gpu-01'   # stable per experiment (station roster / seed / hyperparams)
BASIN   = 'Alto Lerma'
CLAVE   = ''                      # empty -> auto-select top-coverage station in BASIN
LOOKBACK = 90
HORIZONS = '1,2,3,5,7'
HIDDEN  = 64
LAYERS  = 1
DROPOUT = 0.0
BATCH_SIZE = 64
EPOCHS  = 30
LR      = 5e-4
SEED    = 20260101
USE_CLIMA = True
UPLOAD  = True                    # push checkpoints + manifest to R2 under paper2/runs/{RUN_ID}

if UPLOAD and not have_r2:
    raise RuntimeError('UPLOAD=True but R2 credentials not present; set them in Colab secrets or flip UPLOAD to False')

print(f'>>> stage : {STAGE}')
print(f'>>> run-id: {RUN_ID}')
print(f'>>> basin : {BASIN}   clave: {CLAVE or "(auto)"}')
print(f'>>> model : hidden={HIDDEN} layers={LAYERS} dropout={DROPOUT} lookback={LOOKBACK}')
print(f'>>> train : epochs={EPOCHS} bs={BATCH_SIZE} lr={LR} seed={SEED}')
print(f'>>> clima : {"on" if USE_CLIMA else "off"}')
print(f'>>> upload: {"on" if UPLOAD else "off"}')

## 7. Run the stage

Re-invocations of the same `RUN_ID` resume from the last checkpoint saved to R2.

In [ ]:
import shlex

extra = []
if CLAVE:
    extra += ['--clave', CLAVE]
extra += [
    '--run-id', RUN_ID,
    '--basin', BASIN,
    '--lookback', str(LOOKBACK),
    '--horizons', HORIZONS,
    '--hidden', str(HIDDEN),
    '--layers', str(LAYERS),
    '--dropout', str(DROPOUT),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--lr', str(LR),
    '--seed', str(SEED),
]
extra += ['--use-clima' if USE_CLIMA else '--no-clima']
if UPLOAD:
    extra += ['--upload-to-r2']

cmd = 'python -u scripts/{stage}.py {args}'.format(
    stage=STAGE,
    args=' '.join(shlex.quote(a) for a in extra),
)
print('$', cmd)

# Use the ! magic (get_ipython().system) so stdout streams to the cell in real
# time. subprocess.run buffers everything and only prints the CompletedProcess
# repr, hiding the [11_train] progress. python -u forces unbuffered output.
get_ipython().system(f'cd /content/hidroxmx-forecasting && {cmd}')

## 8. Resume command

Paste this after the session reconnects to continue from the last checkpoint on R2.

In [ ]:
resume = ' '.join(['python', f'scripts/{STAGE}.py', *extra])
print('Resume command:')
print('  ' + resume)